# Phase 5: Score Improvement — Steps 1, 2, 6

**Goal**: Improve platform score from 0.5222 toward 0.59+

This notebook implements three quick-win improvements from `docs/plan/improvement_plan.md`:
- **Step 1**: Improved Target Encoding (10-fold, higher smoothing, KDTree fallback)
- **Step 2**: More training rounds with lower learning rate
- **Step 6**: Rank normalization post-processing

Baseline: OOF Spearman 0.5880, Platform 0.5222

In [1]:
import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
from sklearn.model_selection import KFold
from sklearn.neighbors import KDTree
from scipy.stats import spearmanr, ks_2samp, rankdata
import warnings
import time
import os

warnings.filterwarnings('ignore')

SEED = 42
N_FOLDS = 5
DATA_DIR = '../data/'
MODEL_DIR = '../models/'
SUBMIT_DIR = '../submissions/'

np.random.seed(SEED)
print("Setup complete.")

Setup complete.


In [2]:
# Load tier-2 feature data from Phase 2
train_df = pd.read_parquet(f'{DATA_DIR}train_features_tier2.parquet')
test_df = pd.read_parquet(f'{DATA_DIR}test_features_tier2.parquet')

print(f"Train: {train_df.shape}")
print(f"Test:  {test_df.shape}")

# Record old TE stats for comparison
old_grid_te_ks = ks_2samp(train_df['grid_te'], test_df['grid_te']).statistic
old_gp_te_ks = ks_2samp(train_df['grid_period_te'], test_df['grid_period_te']).statistic
print(f"\nOld grid_te       — Train mean: {train_df['grid_te'].mean():.4f}, "
      f"Test mean: {test_df['grid_te'].mean():.4f}, KS: {old_grid_te_ks:.6f}")
print(f"Old grid_period_te — Train mean: {train_df['grid_period_te'].mean():.4f}, "
      f"Test mean: {test_df['grid_period_te'].mean():.4f}, KS: {old_gp_te_ks:.6f}")

Train: (6076546, 31)
Test:  (2028750, 30)

Old grid_te       — Train mean: 0.5020, Test mean: 0.5017, KS: 0.011714
Old grid_period_te — Train mean: 0.5007, Test mean: 0.4996, KS: 0.010233


## Step 1: Improved Target Encoding

Changes from Phase 2 baseline:

| Parameter | Old | New | Why |
|-----------|-----|-----|-----|
| K-Fold splits | 5 | 10 | Train encoding uses 90% data (closer to test's 100%) |
| grid_te smoothing | 30 | 100 | Stronger regularization, reduces variance |
| grid_period_te smoothing | 50 | 150 | Stronger regularization for finer groups |
| Unseen grid fallback | Global mean | KDTree nearest neighbor | Preserves spatial locality |

In [3]:
def kfold_target_encode_v2(train_df, test_df, col, target='invalid_ratio',
                           n_splits=10, smooth=100):
    """
    Improved K-Fold Target Encoding v2.

    Key changes from v1:
    - 10-fold CV: encoding uses 90% of data (vs 80% in 5-fold), reducing
      the distribution gap between train and test TE values
    - Higher smoothing shrinks small-sample estimates toward global mean
    - Returns NaN for unseen test categories (for KDTree fallback later)
    """
    global_mean = float(train_df[target].mean())
    train_encoded = np.full(len(train_df), np.nan, dtype='float64')

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=SEED)

    for fold_train_idx, fold_val_idx in kf.split(train_df):
        fold_data = train_df.iloc[fold_train_idx]
        fold_stats = fold_data.groupby(col)[target].agg(['mean', 'count'])
        # Bayesian smoothing: small samples shrink toward global mean
        smoothed = (
            fold_stats['count'] * fold_stats['mean'] + smooth * global_mean
        ) / (fold_stats['count'] + smooth)
        mapped_vals = train_df.iloc[fold_val_idx][col].map(smoothed).values
        train_encoded[fold_val_idx] = mapped_vals

    # Fill rare edge-case NaN in train (categories appearing in only one fold)
    nan_mask = np.isnan(train_encoded)
    if nan_mask.sum() > 0:
        train_encoded[nan_mask] = global_mean
        print(f"  Filled {nan_mask.sum()} train NaN with global mean")

    # Test: use full training set stats; unseen categories stay NaN
    full_stats = train_df.groupby(col)[target].agg(['mean', 'count'])
    full_smoothed = (
        full_stats['count'] * full_stats['mean'] + smooth * global_mean
    ) / (full_stats['count'] + smooth)
    test_encoded = test_df[col].map(full_smoothed).values.astype('float64')

    n_unseen = np.isnan(test_encoded).sum()
    n_cat = len(full_smoothed)
    print(f"  {col}: {n_cat} categories encoded, "
          f"test has {n_unseen:,} unseen rows ({n_unseen/len(test_df)*100:.1f}%)")

    encoding_map = full_smoothed.to_dict()
    return train_encoded.astype('float32'), test_encoded.astype('float32'), encoding_map

print("kfold_target_encode_v2 defined.")

kfold_target_encode_v2 defined.


In [4]:
# Recompute grid_te: 10-fold, smooth=100
print("Computing improved grid_te (10-fold, smooth=100)...")
train_grid_te, test_grid_te, grid_te_map = kfold_target_encode_v2(
    train_df, test_df, col='grid_id', target='invalid_ratio',
    n_splits=10, smooth=100
)

# KDTree fallback: replace NaN (unseen grids) with nearest known grid's TE
nan_mask = np.isnan(test_grid_te)
if nan_mask.sum() > 0:
    # Build KDTree from known training grid centroids
    known_grids = train_df.groupby('grid_id').agg(
        grid_lon=('grid_lon', 'first'),
        grid_lat=('grid_lat', 'first')
    )
    tree = KDTree(known_grids[['grid_lon', 'grid_lat']].values)

    # Get unique unseen test grids and their coordinates
    unseen_gids = test_df.loc[nan_mask, 'grid_id'].unique()
    unseen_info = (test_df.loc[test_df['grid_id'].isin(unseen_gids)]
                   .groupby('grid_id')[['grid_lon', 'grid_lat']].first())

    # Query nearest known grid for each unseen grid
    dist, idx = tree.query(unseen_info[['grid_lon', 'grid_lat']].values, k=1)
    nearest_gids = known_grids.index[idx.flatten()]

    # Fill unseen grid TE with nearest known grid's TE
    for unseen_gid, nearest_gid in zip(unseen_info.index, nearest_gids):
        gid_mask = test_df['grid_id'].values == unseen_gid
        test_grid_te[gid_mask] = grid_te_map[nearest_gid]

    print(f"  KDTree: filled {len(unseen_gids)} unseen grids "
          f"({nan_mask.sum():,} rows) with nearest-neighbor TE")
    print(f"  Avg distance to nearest grid: {dist.mean():.6f}")

# Update DataFrame
train_df['grid_te'] = train_grid_te
test_df['grid_te'] = test_grid_te

assert np.isnan(test_grid_te).sum() == 0, "grid_te still has NaN!"
print(f"\nNew grid_te — Train: mean={train_grid_te.mean():.4f}, std={train_grid_te.std():.4f}")
print(f"New grid_te — Test:  mean={test_grid_te.mean():.4f}, std={test_grid_te.std():.4f}")

Computing improved grid_te (10-fold, smooth=100)...
  Filled 72 train NaN with global mean
  grid_id: 742 categories encoded, test has 31 unseen rows (0.0%)
  KDTree: filled 21 unseen grids (31 rows) with nearest-neighbor TE
  Avg distance to nearest grid: 2.586162

New grid_te — Train: mean=0.5016, std=0.1151
New grid_te — Test:  mean=0.5013, std=0.1138


In [5]:
# Recompute grid_period_te: 10-fold, smooth=150
print("Computing improved grid_period_te (10-fold, smooth=150)...")
train_gp_te, test_gp_te, gp_te_map = kfold_target_encode_v2(
    train_df, test_df, col='grid_period', target='invalid_ratio',
    n_splits=10, smooth=150
)

# Fallback for unseen grid_period combinations:
# Use the corresponding grid_te value (already KDTree-filled)
nan_mask = np.isnan(test_gp_te)
if nan_mask.sum() > 0:
    test_gp_te[nan_mask] = test_grid_te[nan_mask]
    print(f"  Filled {nan_mask.sum():,} unseen grid_period_te rows with grid_te fallback")

# Update DataFrame
train_df['grid_period_te'] = train_gp_te
test_df['grid_period_te'] = test_gp_te

assert np.isnan(test_gp_te).sum() == 0, "grid_period_te still has NaN!"
print(f"\nNew grid_period_te — Train: mean={train_gp_te.mean():.4f}, std={train_gp_te.std():.4f}")
print(f"New grid_period_te — Test:  mean={test_gp_te.mean():.4f}, std={test_gp_te.std():.4f}")

Computing improved grid_period_te (10-fold, smooth=150)...
  Filled 253 train NaN with global mean
  grid_period: 3114 categories encoded, test has 162 unseen rows (0.0%)
  Filled 162 unseen grid_period_te rows with grid_te fallback

New grid_period_te — Train: mean=0.4992, std=0.1090
New grid_period_te — Test:  mean=0.4982, std=0.1076


In [6]:
# Verify distribution shift is reduced
print("=== TE Distribution Shift Check ===\n")

new_grid_ks = ks_2samp(train_df['grid_te'], test_df['grid_te']).statistic
new_gp_ks = ks_2samp(train_df['grid_period_te'], test_df['grid_period_te']).statistic

print(f"{'Metric':<20s} {'Old KS':>10s} {'New KS':>10s} {'Change':>10s} {'Status':>8s}")
print("-" * 60)
print(f"{'grid_te':<20s} {old_grid_te_ks:>10.6f} {new_grid_ks:>10.6f} "
      f"{new_grid_ks - old_grid_te_ks:>+10.6f} "
      f"{'PASS' if new_grid_ks < 0.005 else 'CHECK':>8s}")
print(f"{'grid_period_te':<20s} {old_gp_te_ks:>10.6f} {new_gp_ks:>10.6f} "
      f"{new_gp_ks - old_gp_te_ks:>+10.6f} "
      f"{'PASS' if new_gp_ks < 0.005 else 'CHECK':>8s}")
print(f"\nTarget: KS stat < 0.005")

=== TE Distribution Shift Check ===

Metric                   Old KS     New KS     Change   Status
------------------------------------------------------------
grid_te                0.011714   0.010623  -0.001091    CHECK
grid_period_te         0.010233   0.009619  -0.000614    CHECK

Target: KS stat < 0.005


## Step 2: Model Training — More Rounds & Lower Learning Rate

Changes from Phase 3 baseline:

| Parameter | Old | New | Why |
|-----------|-----|-----|-----|
| n_estimators | 3000 | 8000 | Both models hit ceiling without early stopping |
| learning_rate | 0.05 | 0.03 | Slower learning with more rounds = better generalization |
| early_stopping_rounds | 50 | 100 | More patience for the lower learning rate |

In [7]:
TARGET = 'invalid_ratio'
EXCLUDE_COLS = ['invalid_ratio', 'grid_lon', 'grid_lat', 'grid_id', 'grid_period']
FEATURES = [c for c in train_df.columns if c not in EXCLUDE_COLS]

print(f"Features ({len(FEATURES)}):")
for i, f in enumerate(FEATURES):
    print(f"  {i+1:2d}. {f}")

X = train_df[FEATURES]
y = train_df[TARGET]
X_test = test_df[FEATURES]

print(f"\nX: {X.shape}, y: {y.shape}, X_test: {X_test.shape}")

Features (26):
   1. total_count
   2. longitude_scaled
   3. latitude_scaled
   4. Precipitations
   5. HauteurNeige
   6. Temperature
   7. ForceVent
   8. day_of_week
   9. month_of_year
  10. hour
  11. log_total_count
  12. count_bin
  13. hour_sin
  14. hour_cos
  15. dow_sin
  16. dow_cos
  17. month_sin
  18. month_cos
  19. grid_te
  20. time_period
  21. grid_period_te
  22. is_raining
  23. has_snow
  24. grid_avg_count
  25. grid_violation_std
  26. grid_sample_count

X: (6076546, 26), y: (6076546,), X_test: (2028750, 26)


In [8]:
# LightGBM: lower lr + more rounds
lgb_params = {
    'objective': 'regression',
    'metric': 'l2',
    'boosting_type': 'gbdt',
    'num_leaves': 31,
    'learning_rate': 0.03,          # was 0.05
    'n_estimators': 8000,           # was 3000
    'reg_lambda': 1.0,
    'min_child_samples': 50,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'verbose': -1,
    'n_jobs': 8,
    'random_state': SEED,
}

kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
lgb_oof = np.zeros(len(train_df))
lgb_test = np.zeros(len(test_df))
lgb_scores = []

print("=== LightGBM Training (lr=0.03, max 8000 rounds) ===\n")
t0 = time.time()

for fold, (tr_idx, va_idx) in enumerate(kf.split(X)):
    X_tr, y_tr = X.iloc[tr_idx], y.iloc[tr_idx]
    X_va, y_va = X.iloc[va_idx], y.iloc[va_idx]

    model = lgb.LGBMRegressor(**lgb_params)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_va, y_va)],
        callbacks=[
            lgb.early_stopping(stopping_rounds=100),  # was 50
            lgb.log_evaluation(period=500),
        ]
    )

    va_pred = model.predict(X_va)
    lgb_oof[va_idx] = va_pred
    lgb_test += model.predict(X_test) / N_FOLDS

    fold_rho = spearmanr(y_va, va_pred)[0]
    lgb_scores.append(fold_rho)
    print(f"  Fold {fold}: Spearman={fold_rho:.4f}, best_iter={model.best_iteration_}")

lgb_oof_rho = spearmanr(y, lgb_oof)[0]
elapsed = time.time() - t0
print(f"\nLGB OOF Spearman: {lgb_oof_rho:.4f}  (baseline: 0.5815)")
print(f"Fold scores: {[f'{s:.4f}' for s in lgb_scores]}")
print(f"Time: {elapsed/60:.1f} min")

=== LightGBM Training (lr=0.03, max 8000 rounds) ===

Training until validation scores don't improve for 100 rounds
[500]	valid_0's l2: 0.100433
[1000]	valid_0's l2: 0.0959664
[1500]	valid_0's l2: 0.0932733
[2000]	valid_0's l2: 0.0914055
[2500]	valid_0's l2: 0.0899945
[3000]	valid_0's l2: 0.0888733
[3500]	valid_0's l2: 0.0880025
[4000]	valid_0's l2: 0.0872592
[4500]	valid_0's l2: 0.0866221
[5000]	valid_0's l2: 0.0860593
[5500]	valid_0's l2: 0.0855793
[6000]	valid_0's l2: 0.085099
[6500]	valid_0's l2: 0.0847477
[7000]	valid_0's l2: 0.0844115
[7500]	valid_0's l2: 0.0841028
[8000]	valid_0's l2: 0.0838129
Did not meet early stopping. Best iteration is:
[8000]	valid_0's l2: 0.0838129
  Fold 0: Spearman=0.5974, best_iter=8000
Training until validation scores don't improve for 100 rounds
[500]	valid_0's l2: 0.100523
[1000]	valid_0's l2: 0.0959837
[1500]	valid_0's l2: 0.0934171
[2000]	valid_0's l2: 0.0916249
[2500]	valid_0's l2: 0.0900431
[3000]	valid_0's l2: 0.0889518
[3500]	valid_0's l2: 0.0

In [9]:
# XGBoost: lower lr + more rounds
xgb_params = {
    'objective': 'reg:squarederror',
    'tree_method': 'hist',
    'max_depth': 6,
    'learning_rate': 0.03,          # was 0.05
    'n_estimators': 8000,           # was 3000
    'reg_lambda': 1.0,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'min_child_weight': 50,
    'random_state': SEED,
    'n_jobs': 8,
    'verbosity': 0,
    'early_stopping_rounds': 100,   # was 50
}

kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
xgb_oof = np.zeros(len(train_df))
xgb_test = np.zeros(len(test_df))
xgb_scores = []

print("=== XGBoost Training (lr=0.03, max 8000 rounds) ===\n")
t0 = time.time()

for fold, (tr_idx, va_idx) in enumerate(kf.split(X)):
    X_tr, y_tr = X.iloc[tr_idx], y.iloc[tr_idx]
    X_va, y_va = X.iloc[va_idx], y.iloc[va_idx]

    model = xgb.XGBRegressor(**xgb_params)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_va, y_va)],
        verbose=500,
    )

    va_pred = model.predict(X_va)
    xgb_oof[va_idx] = va_pred
    xgb_test += model.predict(X_test) / N_FOLDS

    fold_rho = spearmanr(y_va, va_pred)[0]
    xgb_scores.append(fold_rho)
    print(f"  Fold {fold}: Spearman={fold_rho:.4f}, best_iter={model.best_iteration}")

xgb_oof_rho = spearmanr(y, xgb_oof)[0]
elapsed = time.time() - t0
print(f"\nXGB OOF Spearman: {xgb_oof_rho:.4f}  (baseline: 0.5870)")
print(f"Fold scores: {[f'{s:.4f}' for s in xgb_scores]}")
print(f"Time: {elapsed/60:.1f} min")

=== XGBoost Training (lr=0.03, max 8000 rounds) ===

[0]	validation_0-rmse:0.36644
[500]	validation_0-rmse:0.31574
[1000]	validation_0-rmse:0.30807
[1500]	validation_0-rmse:0.30358
[2000]	validation_0-rmse:0.30040
[2500]	validation_0-rmse:0.29816
[3000]	validation_0-rmse:0.29636
[3500]	validation_0-rmse:0.29499
[4000]	validation_0-rmse:0.29386
[4500]	validation_0-rmse:0.29290
[5000]	validation_0-rmse:0.29201
[5500]	validation_0-rmse:0.29126
[6000]	validation_0-rmse:0.29061
[6500]	validation_0-rmse:0.29003
[7000]	validation_0-rmse:0.28953
[7500]	validation_0-rmse:0.28905
[7999]	validation_0-rmse:0.28858
  Fold 0: Spearman=0.6000, best_iter=7999
[0]	validation_0-rmse:0.36626
[500]	validation_0-rmse:0.31540
[1000]	validation_0-rmse:0.30782
[1500]	validation_0-rmse:0.30335
[2000]	validation_0-rmse:0.30029
[2500]	validation_0-rmse:0.29804
[3000]	validation_0-rmse:0.29636
[3500]	validation_0-rmse:0.29498
[4000]	validation_0-rmse:0.29383
[4500]	validation_0-rmse:0.29284
[5000]	validation_0-rm

In [10]:
# Find optimal ensemble weights via grid search on OOF Spearman
print("=== Ensemble Optimization ===\n")

best_w, best_rho = 0, 0
for w in np.arange(0.0, 1.01, 0.05):
    blend = w * lgb_oof + (1 - w) * xgb_oof
    rho = spearmanr(y, blend)[0]
    if rho > best_rho:
        best_w, best_rho = w, rho

print(f"Optimal LGB weight: {best_w:.2f}")
print(f"Ensemble OOF Spearman: {best_rho:.4f}  (baseline: 0.5880)")

# Inter-model correlation (lower = more ensemble diversity)
model_corr = np.corrcoef(lgb_oof, xgb_oof)[0, 1]
print(f"LGB-XGB OOF correlation: {model_corr:.4f}  (baseline: 0.9778)")

# Generate ensemble test predictions
ensemble_test = best_w * lgb_test + (1 - best_w) * xgb_test
ensemble_test = np.clip(ensemble_test, 0, 1)
print(f"\nEnsemble test — mean: {ensemble_test.mean():.4f}, "
      f"range: [{ensemble_test.min():.4f}, {ensemble_test.max():.4f}]")

=== Ensemble Optimization ===

Optimal LGB weight: 0.35
Ensemble OOF Spearman: 0.6012  (baseline: 0.5880)
LGB-XGB OOF correlation: 0.9775  (baseline: 0.9778)

Ensemble test — mean: 0.5353, range: [0.0331, 1.0000]


## Step 6: Rank Normalization Post-Processing

Spearman correlation only cares about **ranking**, not absolute values.
Rank normalization maps test predictions to match the training target distribution
while preserving the prediction order — a monotonic transform that **cannot hurt** Spearman.

In [11]:
# Rank normalization: map test predictions to training target distribution
print("=== Rank Normalization ===\n")

print(f"Before — mean: {ensemble_test.mean():.4f}, std: {ensemble_test.std():.4f}")

# Convert test predictions to percentile ranks, then interpolate
# onto the training target distribution
test_ranks = rankdata(ensemble_test) / len(ensemble_test)
train_sorted = np.sort(y.values)
normalized_test = np.interp(
    test_ranks,
    np.linspace(0, 1, len(train_sorted)),
    train_sorted
)

print(f"After  — mean: {normalized_test.mean():.4f}, std: {normalized_test.std():.4f}")
print(f"Train  — mean: {y.mean():.4f}, std: {y.std():.4f}")

# Verify ranking is preserved (monotonic transform)
rank_corr = spearmanr(ensemble_test, normalized_test)[0]
print(f"\nSpearman(before, after): {rank_corr:.6f}  (should be ~1.0)")

=== Rank Normalization ===

Before — mean: 0.5353, std: 0.2010
After  — mean: 0.5024, std: 0.3684
Train  — mean: 0.5024, std: 0.3684

Spearman(before, after): 0.988190  (should be ~1.0)


In [12]:
# Save OOF and test predictions
np.save(f'{MODEL_DIR}lgb_oof_v2.npy', lgb_oof)
np.save(f'{MODEL_DIR}lgb_test_v2.npy', lgb_test)
np.save(f'{MODEL_DIR}xgb_oof_v2.npy', xgb_oof)
np.save(f'{MODEL_DIR}xgb_test_v2.npy', xgb_test)

# Submission: without rank normalization
sub_raw = pd.DataFrame({'invalid_ratio': ensemble_test})
sub_raw.to_csv(f'{SUBMIT_DIR}ensemble_v2_raw.csv', index=True, index_label='')

# Submission: with rank normalization
sub_norm = pd.DataFrame({'invalid_ratio': normalized_test})
sub_norm.to_csv(f'{SUBMIT_DIR}ensemble_v2_ranked.csv', index=True, index_label='')

print("Saved:")
print(f"  Predictions: {MODEL_DIR}[lgb|xgb]_[oof|test]_v2.npy")
print(f"  Submission:  {SUBMIT_DIR}ensemble_v2_raw.csv    (without rank norm)")
print(f"  Submission:  {SUBMIT_DIR}ensemble_v2_ranked.csv (with rank norm)")

Saved:
  Predictions: ../models/[lgb|xgb]_[oof|test]_v2.npy
  Submission:  ../submissions/ensemble_v2_raw.csv    (without rank norm)
  Submission:  ../submissions/ensemble_v2_ranked.csv (with rank norm)


In [13]:
# Final validation
print("=== Pre-Submission Validation ===\n")

checks = [
    ('LGB OOF Spearman',  f'{lgb_oof_rho:.4f}  (was 0.5815)', 'PASS' if lgb_oof_rho > 0.58 else 'CHECK'),
    ('XGB OOF Spearman',  f'{xgb_oof_rho:.4f}  (was 0.5870)', 'PASS' if xgb_oof_rho > 0.58 else 'CHECK'),
    ('Ensemble OOF',      f'{best_rho:.4f}  (was 0.5880)', 'PASS' if best_rho > 0.58 else 'CHECK'),
    ('NaN (raw)',          str(np.isnan(ensemble_test).sum()), 'PASS' if np.isnan(ensemble_test).sum() == 0 else 'FAIL'),
    ('NaN (ranked)',       str(np.isnan(normalized_test).sum()), 'PASS' if np.isnan(normalized_test).sum() == 0 else 'FAIL'),
    ('Row count',          str(len(ensemble_test)), 'PASS' if len(ensemble_test) == 2028750 else 'FAIL'),
    ('Range (raw)',        f'[{ensemble_test.min():.4f}, {ensemble_test.max():.4f}]',
                           'PASS' if ensemble_test.min() >= 0 and ensemble_test.max() <= 1 else 'FAIL'),
    ('Range (ranked)',     f'[{normalized_test.min():.4f}, {normalized_test.max():.4f}]',
                           'PASS' if normalized_test.min() >= 0 and normalized_test.max() <= 1 else 'FAIL'),
]

for name, val, status in checks:
    print(f"  {name:<22s} {val:<30s} {status}")

# Summary comparison table
print(f"\n=== Improvement Summary ===")
print(f"{'Metric':<25s} {'Baseline':>10s} {'New':>10s} {'Delta':>10s}")
print("-" * 57)
print(f"{'LGB OOF Spearman':<25s} {'0.5815':>10s} {lgb_oof_rho:>10.4f} {lgb_oof_rho - 0.5815:>+10.4f}")
print(f"{'XGB OOF Spearman':<25s} {'0.5870':>10s} {xgb_oof_rho:>10.4f} {xgb_oof_rho - 0.5870:>+10.4f}")
print(f"{'Ensemble OOF Spearman':<25s} {'0.5880':>10s} {best_rho:>10.4f} {best_rho - 0.5880:>+10.4f}")
print(f"{'LGB-XGB Correlation':<25s} {'0.9778':>10s} {model_corr:>10.4f} {model_corr - 0.9778:>+10.4f}")
print(f"{'grid_te KS stat':<25s} {old_grid_te_ks:>10.6f} {new_grid_ks:>10.6f} {new_grid_ks - old_grid_te_ks:>+10.6f}")
print(f"{'grid_period_te KS stat':<25s} {old_gp_te_ks:>10.6f} {new_gp_ks:>10.6f} {new_gp_ks - old_gp_te_ks:>+10.6f}")

print(f"\nRecommended submission: {SUBMIT_DIR}ensemble_v2_ranked.csv")
print("\nNext steps: Step 3 (Optuna tuning), Step 4 (CatBoost), Step 5 (Stacking)")

=== Pre-Submission Validation ===

  LGB OOF Spearman       0.5959  (was 0.5815)           PASS
  XGB OOF Spearman       0.5994  (was 0.5870)           PASS
  Ensemble OOF           0.6012  (was 0.5880)           PASS
  NaN (raw)              0                              PASS
  NaN (ranked)           0                              PASS
  Row count              2028750                        PASS
  Range (raw)            [0.0331, 1.0000]               PASS
  Range (ranked)         [0.0000, 1.0000]               PASS

=== Improvement Summary ===
Metric                      Baseline        New      Delta
---------------------------------------------------------
LGB OOF Spearman              0.5815     0.5959    +0.0144
XGB OOF Spearman              0.5870     0.5994    +0.0124
Ensemble OOF Spearman         0.5880     0.6012    +0.0132
LGB-XGB Correlation           0.9778     0.9775    -0.0003
grid_te KS stat             0.011714   0.010623  -0.001091
grid_period_te KS stat      0.010233

---

## Step 3: Optuna Hyperparameter Tuning

- **Subsample**: 1M rows from 6M for fast trial evaluation
- **CV**: 3-Fold, Spearman correlation as objective
- **Trials**: 50 per model
- After tuning, retrain on full 6M data with 5-Fold CV

Estimated time: ~20-40 min per model for Optuna, then ~1-2 hours for full retraining.

In [14]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
print(f"Optuna {optuna.__version__}")

# Subsample 1M rows for fast Optuna trials
sub_df = train_df.sample(n=1_000_000, random_state=SEED)
sub_X = sub_df[FEATURES]
sub_y = sub_df[TARGET]
print(f"Subsample: {sub_X.shape}")

Optuna 4.8.0
Subsample: (1000000, 26)


In [15]:
def lgb_objective(trial):
    params = {
        'objective': 'regression',
        'metric': 'l2',
        'boosting_type': 'gbdt',
        'num_leaves': trial.suggest_int('num_leaves', 15, 127),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'min_child_samples': trial.suggest_int('min_child_samples', 20, 200),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.1, 10.0, log=True),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 5.0),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.5, 1.0),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.5, 1.0),
        'bagging_freq': 5,
        'verbose': -1,
        'n_jobs': 8,
        'random_state': SEED,
    }

    kf_opt = KFold(n_splits=3, shuffle=True, random_state=SEED)
    scores = []
    for tr_idx, va_idx in kf_opt.split(sub_X):
        model = lgb.LGBMRegressor(n_estimators=3000, **params)
        model.fit(
            sub_X.iloc[tr_idx], sub_y.iloc[tr_idx],
            eval_set=[(sub_X.iloc[va_idx], sub_y.iloc[va_idx])],
            callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)],
        )
        pred = model.predict(sub_X.iloc[va_idx])
        scores.append(spearmanr(sub_y.iloc[va_idx], pred)[0])

    return np.mean(scores)

print("Running Optuna LGB tuning (50 trials)...")
t0 = time.time()
lgb_study = optuna.create_study(direction='maximize', study_name='lgb_tuning')
lgb_study.optimize(lgb_objective, n_trials=50, show_progress_bar=True)

print(f"\nDone in {(time.time()-t0)/60:.1f} min")
print(f"Best LGB Spearman (subsample): {lgb_study.best_value:.4f}")
print(f"Best params: {lgb_study.best_params}")

Running Optuna LGB tuning (50 trials)...


  0%|          | 0/50 [00:00<?, ?it/s]

Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[3000]	valid_0's l2: 0.0931526
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[2999]	valid_0's l2: 0.0935741
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[3000]	valid_0's l2: 0.0932044


Best trial: 0. Best value: 0.534381:   2%|▏         | 1/50 [02:15<1:50:44, 135.60s/it]

Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[3000]	valid_0's l2: 0.0939216
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[3000]	valid_0's l2: 0.0944386
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[3000]	valid_0's l2: 0.0941801


...

Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[1913]	valid_0's l2: 0.086036
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[2247]	valid_0's l2: 0.0863069
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[2650]	valid_0's l2: 0.0859926


Best trial: 47. Best value: 0.585312:  98%|█████████▊| 49/50 [2:36:04<03:07, 187.83s/it]

Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[2866]	valid_0's l2: 0.0860048
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[2990]	valid_0's l2: 0.0862277
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[2998]	valid_0's l2: 0.086102


Best trial: 47. Best value: 0.585312: 100%|██████████| 50/50 [2:40:21<00:00, 192.43s/it]


Done in 160.4 min
Best LGB Spearman (subsample): 0.5853
Best params: {'num_leaves': 100, 'learning_rate': 0.056381501348435746, 'min_child_samples': 69, 'reg_lambda': 0.4524558370492589, 'reg_alpha': 1.242953508157269, 'feature_fraction': 0.8437282229785125, 'bagging_fraction': 0.9723892542828807}


In [16]:
def xgb_objective(trial):
    params = {
        'objective': 'reg:squarederror',
        'tree_method': 'hist',
        'max_depth': trial.suggest_int('max_depth', 4, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'min_child_weight': trial.suggest_int('min_child_weight', 10, 200),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.1, 10.0, log=True),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 5.0),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'random_state': SEED,
        'n_jobs': 8,
        'verbosity': 0,
        'early_stopping_rounds': 50,
    }

    kf_opt = KFold(n_splits=3, shuffle=True, random_state=SEED)
    scores = []
    for tr_idx, va_idx in kf_opt.split(sub_X):
        model = xgb.XGBRegressor(n_estimators=3000, **params)
        model.fit(
            sub_X.iloc[tr_idx], sub_y.iloc[tr_idx],
            eval_set=[(sub_X.iloc[va_idx], sub_y.iloc[va_idx])],
            verbose=0,
        )
        pred = model.predict(sub_X.iloc[va_idx])
        scores.append(spearmanr(sub_y.iloc[va_idx], pred)[0])

    return np.mean(scores)

print("Running Optuna XGB tuning (50 trials)...")
t0 = time.time()
xgb_study = optuna.create_study(direction='maximize', study_name='xgb_tuning')
xgb_study.optimize(xgb_objective, n_trials=50, show_progress_bar=True)

print(f"\nDone in {(time.time()-t0)/60:.1f} min")
print(f"Best XGB Spearman (subsample): {xgb_study.best_value:.4f}")
print(f"Best params: {xgb_study.best_params}")

Running Optuna XGB tuning (50 trials)...


Best trial: 28. Best value: 0.590142: 100%|██████████| 50/50 [2:11:47<00:00, 158.14s/it]


Done in 131.8 min
Best XGB Spearman (subsample): 0.5901
Best params: {'max_depth': 10, 'learning_rate': 0.036225313048528045, 'min_child_weight': 11, 'reg_lambda': 1.560688696096005, 'reg_alpha': 1.2385547133219446, 'subsample': 0.9475358514709203, 'colsample_bytree': 0.9505250929239788}


In [17]:
print("=== Optuna Best Parameters ===\n")

print("LightGBM:")
for k, v in lgb_study.best_params.items():
    print(f"  {k}: {v}")
print(f"  Best subsample Spearman: {lgb_study.best_value:.4f}")

print(f"\nXGBoost:")
for k, v in xgb_study.best_params.items():
    print(f"  {k}: {v}")
print(f"  Best subsample Spearman: {xgb_study.best_value:.4f}")

=== Optuna Best Parameters ===

LightGBM:
  num_leaves: 100
  learning_rate: 0.056381501348435746
  min_child_samples: 69
  reg_lambda: 0.4524558370492589
  reg_alpha: 1.242953508157269
  feature_fraction: 0.8437282229785125
  bagging_fraction: 0.9723892542828807
  Best subsample Spearman: 0.5853

XGBoost:
  max_depth: 10
  learning_rate: 0.036225313048528045
  min_child_weight: 11
  reg_lambda: 1.560688696096005
  reg_alpha: 1.2385547133219446
  subsample: 0.9475358514709203
  colsample_bytree: 0.9505250929239788
  Best subsample Spearman: 0.5901


### Retrain with Tuned Parameters on Full Data

Using Optuna-found parameters with more estimators (10000) and longer early stopping patience (150).

In [18]:
# Build final LGB params from Optuna results
lgb_tuned = lgb_study.best_params.copy()
lgb_tuned.update({
    'objective': 'regression',
    'metric': 'l2',
    'boosting_type': 'gbdt',
    'bagging_freq': 5,
    'verbose': -1,
    'n_jobs': 8,
    'random_state': SEED,
    'n_estimators': 10000,
})

kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
lgb_oof_v3 = np.zeros(len(train_df))
lgb_test_v3 = np.zeros(len(test_df))
lgb_scores_v3 = []

print("=== LGB Retrain with Tuned Params (full data, 5-Fold) ===\n")
t0 = time.time()

for fold, (tr_idx, va_idx) in enumerate(kf.split(X)):
    X_tr, y_tr = X.iloc[tr_idx], y.iloc[tr_idx]
    X_va, y_va = X.iloc[va_idx], y.iloc[va_idx]

    model = lgb.LGBMRegressor(**lgb_tuned)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_va, y_va)],
        callbacks=[
            lgb.early_stopping(stopping_rounds=150),
            lgb.log_evaluation(period=500),
        ]
    )

    va_pred = model.predict(X_va)
    lgb_oof_v3[va_idx] = va_pred
    lgb_test_v3 += model.predict(X_test) / N_FOLDS

    fold_rho = spearmanr(y_va, va_pred)[0]
    lgb_scores_v3.append(fold_rho)
    print(f"  Fold {fold}: Spearman={fold_rho:.4f}, best_iter={model.best_iteration_}")

lgb_oof_v3_rho = spearmanr(y, lgb_oof_v3)[0]
elapsed = time.time() - t0
print(f"\nLGB v3 OOF Spearman: {lgb_oof_v3_rho:.4f}  (v2: {lgb_oof_rho:.4f})")
print(f"Fold scores: {[f'{s:.4f}' for s in lgb_scores_v3]}")
print(f"Time: {elapsed/60:.1f} min")

=== LGB Retrain with Tuned Params (full data, 5-Fold) ===

Training until validation scores don't improve for 150 rounds
[500]	valid_0's l2: 0.0885934
[1000]	valid_0's l2: 0.0846902
[1500]	valid_0's l2: 0.0828427
[2000]	valid_0's l2: 0.081796
[2500]	valid_0's l2: 0.0809999
[3000]	valid_0's l2: 0.0803602
[3500]	valid_0's l2: 0.0798753
[4000]	valid_0's l2: 0.0795414
[4500]	valid_0's l2: 0.0792442
[5000]	valid_0's l2: 0.0790221
[5500]	valid_0's l2: 0.0788254
[6000]	valid_0's l2: 0.0786459
[6500]	valid_0's l2: 0.0785044
[7000]	valid_0's l2: 0.0783517
[7500]	valid_0's l2: 0.078237
[8000]	valid_0's l2: 0.078121
[8500]	valid_0's l2: 0.0780045
[9000]	valid_0's l2: 0.0779017
[9500]	valid_0's l2: 0.0777981
[10000]	valid_0's l2: 0.0777319
Did not meet early stopping. Best iteration is:
[9999]	valid_0's l2: 0.0777317
  Fold 0: Spearman=0.6333, best_iter=9999
Training until validation scores don't improve for 150 rounds
[500]	valid_0's l2: 0.0885292
[1000]	valid_0's l2: 0.0847106
[1500]	valid_0's l

In [19]:
# Build final XGB params from Optuna results
xgb_tuned = xgb_study.best_params.copy()
xgb_tuned.update({
    'objective': 'reg:squarederror',
    'tree_method': 'hist',
    'random_state': SEED,
    'n_jobs': 8,
    'verbosity': 0,
    'n_estimators': 10000,
    'early_stopping_rounds': 150,
})

kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
xgb_oof_v3 = np.zeros(len(train_df))
xgb_test_v3 = np.zeros(len(test_df))
xgb_scores_v3 = []

print("=== XGB Retrain with Tuned Params (full data, 5-Fold) ===\n")
t0 = time.time()

for fold, (tr_idx, va_idx) in enumerate(kf.split(X)):
    X_tr, y_tr = X.iloc[tr_idx], y.iloc[tr_idx]
    X_va, y_va = X.iloc[va_idx], y.iloc[va_idx]

    model = xgb.XGBRegressor(**xgb_tuned)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_va, y_va)],
        verbose=500,
    )

    va_pred = model.predict(X_va)
    xgb_oof_v3[va_idx] = va_pred
    xgb_test_v3 += model.predict(X_test) / N_FOLDS

    fold_rho = spearmanr(y_va, va_pred)[0]
    xgb_scores_v3.append(fold_rho)
    print(f"  Fold {fold}: Spearman={fold_rho:.4f}, best_iter={model.best_iteration}")

xgb_oof_v3_rho = spearmanr(y, xgb_oof_v3)[0]
elapsed = time.time() - t0
print(f"\nXGB v3 OOF Spearman: {xgb_oof_v3_rho:.4f}  (v2: {xgb_oof_rho:.4f})")
print(f"Fold scores: {[f'{s:.4f}' for s in xgb_scores_v3]}")
print(f"Time: {elapsed/60:.1f} min")

=== XGB Retrain with Tuned Params (full data, 5-Fold) ===

[0]	validation_0-rmse:0.36569
[500]	validation_0-rmse:0.29181
[1000]	validation_0-rmse:0.28483
[1500]	validation_0-rmse:0.28181
[2000]	validation_0-rmse:0.28031
[2500]	validation_0-rmse:0.27930
[3000]	validation_0-rmse:0.27863
[3500]	validation_0-rmse:0.27820
[4000]	validation_0-rmse:0.27789
[4500]	validation_0-rmse:0.27766
[5000]	validation_0-rmse:0.27752
[5500]	validation_0-rmse:0.27743
[6000]	validation_0-rmse:0.27740
[6068]	validation_0-rmse:0.27740
  Fold 0: Spearman=0.6385, best_iter=5918
[0]	validation_0-rmse:0.36554
[500]	validation_0-rmse:0.29162
[1000]	validation_0-rmse:0.28462
[1500]	validation_0-rmse:0.28188
[2000]	validation_0-rmse:0.28039
[2500]	validation_0-rmse:0.27940
[3000]	validation_0-rmse:0.27869
[3500]	validation_0-rmse:0.27827
[4000]	validation_0-rmse:0.27800
[4500]	validation_0-rmse:0.27781
[5000]	validation_0-rmse:0.27768
[5500]	validation_0-rmse:0.27762
[5874]	validation_0-rmse:0.27760
  Fold 1: Spearm

---

## Step 4: CatBoost — Third Model for Ensemble Diversity

LGB-XGB OOF correlation is ~0.977 — too high for meaningful ensemble gains.
CatBoost uses **symmetric trees + ordered boosting**, architecturally different from LGB/XGB.

Key strategy: use `grid_id` and `grid_period` as **native categorical features**.
CatBoost applies ordered target encoding internally, avoiding the TE distribution shift problem.

In [20]:
from catboost import CatBoostRegressor, Pool

# CatBoost features: same 26 + grid_id and grid_period as native categoricals
CB_FEATURES = FEATURES + ['grid_id', 'grid_period']
CAT_COLS = ['grid_id', 'grid_period']
cat_indices = [CB_FEATURES.index(c) for c in CAT_COLS]

X_cb = train_df[CB_FEATURES]
X_test_cb = test_df[CB_FEATURES]

cb_params = {
    'iterations': 8000,
    'learning_rate': 0.05,
    'depth': 6,
    'l2_leaf_reg': 3.0,
    'random_seed': SEED,
    'verbose': 500,
    'thread_count': 8,
    'eval_metric': 'RMSE',
}

kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
cb_oof = np.zeros(len(train_df))
cb_test = np.zeros(len(test_df))
cb_scores = []

print(f"=== CatBoost Training (5-Fold, {len(CB_FEATURES)} features, "
      f"2 native categoricals) ===\n")
t0 = time.time()

for fold, (tr_idx, va_idx) in enumerate(kf.split(X_cb)):
    X_tr = X_cb.iloc[tr_idx]
    y_tr = y.iloc[tr_idx]
    X_va = X_cb.iloc[va_idx]
    y_va = y.iloc[va_idx]

    train_pool = Pool(X_tr, y_tr, cat_features=cat_indices)
    val_pool = Pool(X_va, y_va, cat_features=cat_indices)

    model = CatBoostRegressor(**cb_params)
    model.fit(train_pool, eval_set=val_pool, early_stopping_rounds=100)

    va_pred = model.predict(X_va)
    cb_oof[va_idx] = va_pred

    test_pool = Pool(X_test_cb, cat_features=cat_indices)
    cb_test += model.predict(test_pool) / N_FOLDS

    fold_rho = spearmanr(y_va, va_pred)[0]
    cb_scores.append(fold_rho)
    print(f"  Fold {fold}: Spearman={fold_rho:.4f}, "
          f"best_iter={model.get_best_iteration()}")

cb_oof_rho = spearmanr(y, cb_oof)[0]
elapsed = time.time() - t0
print(f"\nCatBoost OOF Spearman: {cb_oof_rho:.4f}")
print(f"Fold scores: {[f'{s:.4f}' for s in cb_scores]}")
print(f"Time: {elapsed/60:.1f} min")

=== CatBoost Training (5-Fold, 28 features, 2 native categoricals) ===

0:	learn: 0.3650850	test: 0.3653441	best: 0.3653441 (0)	total: 668ms	remaining: 1h 29m 4s
500:	learn: 0.3198140	test: 0.3198852	best: 0.3198852 (500)	total: 3m 40s	remaining: 55m 5s
1000:	learn: 0.3141553	test: 0.3142280	best: 0.3142280 (1000)	total: 7m 52s	remaining: 55m 1s
1500:	learn: 0.3106288	test: 0.3107368	best: 0.3107368 (1500)	total: 12m 10s	remaining: 52m 43s
2000:	learn: 0.3079697	test: 0.3081111	best: 0.3081111 (2000)	total: 16m 40s	remaining: 49m 58s
2500:	learn: 0.3058224	test: 0.3059941	best: 0.3059941 (2500)	total: 21m 19s	remaining: 46m 53s
3000:	learn: 0.3041006	test: 0.3043159	best: 0.3043159 (3000)	total: 25m 57s	remaining: 43m 14s
3500:	learn: 0.3026492	test: 0.3028979	best: 0.3028979 (3500)	total: 30m 41s	remaining: 39m 26s
4000:	learn: 0.3013760	test: 0.3016509	best: 0.3016509 (4000)	total: 35m 20s	remaining: 35m 19s
4500:	learn: 0.3002888	test: 0.3005978	best: 0.3005978 (4500)	total: 40m 5s	

---

## 3-Model Ensemble Optimization

Grid search over LGB, XGB, CatBoost weights to maximize OOF Spearman.
Also checks inter-model correlations — lower correlation = more ensemble benefit.

In [21]:
# Inter-model correlations
corr_lgb_xgb = np.corrcoef(lgb_oof_v3, xgb_oof_v3)[0, 1]
corr_lgb_cb  = np.corrcoef(lgb_oof_v3, cb_oof)[0, 1]
corr_xgb_cb  = np.corrcoef(xgb_oof_v3, cb_oof)[0, 1]

print("=== Inter-Model Correlations ===")
print(f"  LGB-XGB: {corr_lgb_xgb:.4f}")
print(f"  LGB-CB:  {corr_lgb_cb:.4f}")
print(f"  XGB-CB:  {corr_xgb_cb:.4f}")

# Grid search over 3-model weights
best_w1, best_w2, best_rho3 = 0, 0, 0
for w1 in np.arange(0, 1.01, 0.05):
    for w2 in np.arange(0, 1.01 - w1, 0.05):
        w3 = round(1.0 - w1 - w2, 2)
        blend = w1 * lgb_oof_v3 + w2 * xgb_oof_v3 + w3 * cb_oof
        rho = spearmanr(y, blend)[0]
        if rho > best_rho3:
            best_w1, best_w2, best_rho3 = w1, w2, rho

best_w3 = round(1.0 - best_w1 - best_w2, 2)

# Also check best 2-model (LGB+XGB only, for comparison)
best_w_2m, best_rho_2m = 0, 0
for w in np.arange(0, 1.01, 0.05):
    blend = w * lgb_oof_v3 + (1 - w) * xgb_oof_v3
    rho = spearmanr(y, blend)[0]
    if rho > best_rho_2m:
        best_w_2m, best_rho_2m = w, rho

print(f"\n=== Ensemble Results ===")
print(f"  2-model (LGB+XGB):          Spearman={best_rho_2m:.4f}  "
      f"(LGB={best_w_2m:.2f}, XGB={1-best_w_2m:.2f})")
print(f"  3-model (LGB+XGB+CB):       Spearman={best_rho3:.4f}  "
      f"(LGB={best_w1:.2f}, XGB={best_w2:.2f}, CB={best_w3:.2f})")
print(f"  CatBoost ensemble benefit:  {best_rho3 - best_rho_2m:+.4f}")

# Generate final test predictions
ensemble_test_v3 = (best_w1 * lgb_test_v3
                    + best_w2 * xgb_test_v3
                    + best_w3 * cb_test)
ensemble_test_v3 = np.clip(ensemble_test_v3, 0, 1)

print(f"\n  Test predictions — mean: {ensemble_test_v3.mean():.4f}, "
      f"range: [{ensemble_test_v3.min():.4f}, {ensemble_test_v3.max():.4f}]")

=== Inter-Model Correlations ===
  LGB-XGB: 0.9652
  LGB-CB:  0.9330
  XGB-CB:  0.9109

=== Ensemble Results ===
  2-model (LGB+XGB):          Spearman=0.6408  (LGB=0.35, XGB=0.65)
  3-model (LGB+XGB+CB):       Spearman=0.6408  (LGB=0.35, XGB=0.65, CB=-0.00)
  CatBoost ensemble benefit:  +0.0000

  Test predictions — mean: 0.5321, range: [0.0000, 1.0000]


In [22]:
# Save all predictions
np.save(f'{MODEL_DIR}lgb_oof_v3.npy', lgb_oof_v3)
np.save(f'{MODEL_DIR}lgb_test_v3.npy', lgb_test_v3)
np.save(f'{MODEL_DIR}xgb_oof_v3.npy', xgb_oof_v3)
np.save(f'{MODEL_DIR}xgb_test_v3.npy', xgb_test_v3)
np.save(f'{MODEL_DIR}cb_oof.npy', cb_oof)
np.save(f'{MODEL_DIR}cb_test.npy', cb_test)

# Submission file
sub = pd.DataFrame({'invalid_ratio': ensemble_test_v3})
sub.to_csv(f'{SUBMIT_DIR}ensemble_v3.csv', index=True, index_label='')

print("Saved:")
print(f"  Predictions: {MODEL_DIR}[lgb|xgb|cb]_[oof|test]_v3.npy")
print(f"  Submission:  {SUBMIT_DIR}ensemble_v3.csv")

Saved:
  Predictions: ../models/[lgb|xgb|cb]_[oof|test]_v3.npy
  Submission:  ../submissions/ensemble_v3.csv


In [23]:
# Final validation
print("=== Pre-Submission Validation ===\n")

checks = [
    ('LGB v3 OOF',   f'{lgb_oof_v3_rho:.4f}', 'PASS' if lgb_oof_v3_rho > 0.58 else 'CHECK'),
    ('XGB v3 OOF',   f'{xgb_oof_v3_rho:.4f}', 'PASS' if xgb_oof_v3_rho > 0.58 else 'CHECK'),
    ('CB OOF',       f'{cb_oof_rho:.4f}',      'PASS' if cb_oof_rho > 0.55 else 'CHECK'),
    ('3-Model Ens',  f'{best_rho3:.4f}',       'PASS' if best_rho3 > 0.60 else 'CHECK'),
    ('NaN count',    str(np.isnan(ensemble_test_v3).sum()),
                     'PASS' if np.isnan(ensemble_test_v3).sum() == 0 else 'FAIL'),
    ('Row count',    str(len(ensemble_test_v3)),
                     'PASS' if len(ensemble_test_v3) == 2028750 else 'FAIL'),
    ('Range',        f'[{ensemble_test_v3.min():.4f}, {ensemble_test_v3.max():.4f}]',
                     'PASS' if ensemble_test_v3.min() >= 0 and ensemble_test_v3.max() <= 1 else 'FAIL'),
]

for name, val, status in checks:
    print(f"  {name:<18s} {val:<30s} {status}")

# Full improvement history
print(f"\n=== Full Improvement History ===")
print(f"{'Version':<15s} {'LGB':>8s} {'XGB':>8s} {'Ensemble':>10s} {'Platform':>10s}")
print("-" * 55)
print(f"{'v1 (baseline)':<15s} {'0.5815':>8s} {'0.5870':>8s} {'0.5880':>10s} {'0.5222':>10s}")
print(f"{'v2 (Step1+2)':<15s} {lgb_oof_rho:>8.4f} {xgb_oof_rho:>8.4f} {best_rho:>10.4f} {'0.5338':>10s}")
print(f"{'v3 (Step3+4)':<15s} {lgb_oof_v3_rho:>8.4f} {xgb_oof_v3_rho:>8.4f} {best_rho3:>10.4f} {'submit →':>10s}")

print(f"\nSubmit: {SUBMIT_DIR}ensemble_v3.csv")

=== Pre-Submission Validation ===

  LGB v3 OOF         0.6322                         PASS
  XGB v3 OOF         0.6379                         PASS
  CB OOF             0.5728                         PASS
  3-Model Ens        0.6408                         PASS
  NaN count          0                              PASS
  Row count          2028750                        PASS
  Range              [0.0000, 1.0000]               PASS

=== Full Improvement History ===
Version              LGB      XGB   Ensemble   Platform
-------------------------------------------------------
v1 (baseline)     0.5815   0.5870     0.5880     0.5222
v2 (Step1+2)      0.5959   0.5994     0.6012     0.5338
v3 (Step3+4)      0.6322   0.6379     0.6408   submit →

Submit: ../submissions/ensemble_v3.csv
